In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import (
    make_scorer,
    matthews_corrcoef,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import sys
sys.path.append("../../utils/")

from utils import *

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[2]

NOMBRE_EXPERIMENTO = "BCCC17__RUS_SMOTE_ENN__v1__pca4_svm__v1"
CARPETA_DATASET = "BCCC17__RUS_SMOTE_ENN__v1"

NOMBRE_DATASET_TRAIN = f"{CARPETA_DATASET}__train.csv"
NOMBRE_DATASET_TEST = f"{CARPETA_DATASET}__test.csv"

RUTA_DATASET = PROJECT_ROOT / "02_datasets" / "processed" / CARPETA_DATASET
RUTA_RESULTADOS = PROJECT_ROOT / "04_experimentos" / "logs" / "resultados" / NOMBRE_EXPERIMENTO

NOMBRE_RESULTADOS_CV_CSV = f"{NOMBRE_EXPERIMENTO}__folds.csv"
NOMBRE_RESULTADOS_CV_JSON = f"{NOMBRE_EXPERIMENTO}__summary_cv.json"
NOMBRE_RESULTADOS_TEST_JSON = f"{NOMBRE_EXPERIMENTO}__summary_test.json"
NOMBRE_RESULTADOS_TEST_CSV = f"{NOMBRE_EXPERIMENTO}__metricas_test.csv"
NOMBRE_RESULTADOS_TEST_CM_CSV = f"{NOMBRE_EXPERIMENTO}__confusion_matrix_test.csv"

# ===== PARÁMETROS =====
LABEL_COL = "LABEL"

N_SPLITS = 5
SHUFFLE = True
RANDOM_STATE = 42

# ==== CONFIG SVM ====
C = 1.0
KERNEL = "rbf"      # "linear", "rbf", "poly", "sigmoid"
GAMMA = "scale"     # "scale", "auto" o un float

# ===== CONFIG PCA =====
N_COMPONENTS_PCA = 4

In [3]:
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Ruta dataset train:")
print((RUTA_DATASET / NOMBRE_DATASET_TRAIN).resolve())
print()

print("Ruta dataset test:")
print((RUTA_DATASET / NOMBRE_DATASET_TEST).resolve())
print()

print("Ruta resultados:")
print(RUTA_RESULTADOS.resolve())

Ruta dataset train:
/home/javier/TFG_MODELOS_SIMPLES/02_datasets/processed/BCCC17__RUS_SMOTE_ENN__v1/BCCC17__RUS_SMOTE_ENN__v1__train.csv

Ruta dataset test:
/home/javier/TFG_MODELOS_SIMPLES/02_datasets/processed/BCCC17__RUS_SMOTE_ENN__v1/BCCC17__RUS_SMOTE_ENN__v1__test.csv

Ruta resultados:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/BCCC17__RUS_SMOTE_ENN__v1__pca4_svm__v1


In [4]:
df_train = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TRAIN,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset train:")
print(df_train.shape)

df_train.head()

Forma del dataset train:
(131810, 64)


,DST_PORT,PROTOCOL,DURATION,PACKETS_COUNT,FWD_TOTAL_PAYLOAD_BYTES,PAYLOAD_BYTES_MAX,PAYLOAD_BYTES_MIN,PAYLOAD_BYTES_MEAN,PAYLOAD_BYTES_VARIANCE,FWD_PAYLOAD_BYTES_VARIANCE,...,BWD_SYN_FLAG_COUNTS,BWD_CWR_FLAG_COUNTS,BWD_RST_FLAG_COUNTS,PACKETS_IAT_MEAN,FWD_PACKETS_IAT_MEAN,FWD_PACKETS_IAT_STD,BWD_PACKETS_IAT_MEAN,SUBFLOW_FWD_PACKETS,SUBFLOW_FWD_BYTES,LABEL
0,443,0,0.000000,1,0,0,0,0.000000,0.000000,0.00,...,0,0,0,1.499349e+09,1.499349e+09,0.00000,0.000000,0.0,0.0,0
1,443,0,31.768311,35,6254,1752,0,347.571429,302146.987755,256300.01,...,1,0,0,9.343621e-01,1.669347e+00,6.99597,2.265553,20.0,6254.0,0
2,443,0,0.000000,1,0,0,0,0.000000,0.000000,0.00,...,0,0,0,1.499440e+09,1.499440e+09,0.00000,0.000000,0.0,0.0,0
3,53,1,0.000293,4,66,269,33,151.000000,13924.000000,0.00,...,0,0,0,9.759000e-05,4.888000e-05,0.00000,0.000048,0.0,0.0,0
4,53,1,0.000196,4,80,102,40,71.000000,961.000000,0.00,...,0,0,0,6.533000e-05,4.792000e-05,0.00000,0.000004,0.0,0.0,0


In [5]:
if LABEL_COL not in df_train.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en train")

print("Última columna train:", df_train.columns[-1])
print("Tipo de LABEL train:", df_train[LABEL_COL].dtype)
print()

print("Distribución de clases en train:")
display(df_train[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna train: LABEL
Tipo de LABEL train: int64

Distribución de clases en train:


,count
LABEL,
0,10000
8,9991
4,9978
13,9974
2,9973
7,9947
12,9811
9,9620
6,9571


In [6]:
X_train = df_train.drop(columns=[LABEL_COL]).copy()
y_train = df_train[LABEL_COL].copy()

print("Shape X_train:", X_train.shape)
print("Shape y_train:", y_train.shape)

Shape X_train: (131810, 63)
Shape y_train: (131810,)


In [7]:
columnas_no_numericas_train = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_train:")
print(columnas_no_numericas_train)

if len(columnas_no_numericas_train) > 0:
    raise ValueError("Hay columnas no numéricas en X_train. Revísalas antes de seguir.")

Columnas no numéricas en X_train:
[]


In [8]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=N_COMPONENTS_PCA)),
    ("svm", SVC(
        C=C,
        kernel=KERNEL,
        gamma=GAMMA,
        class_weight="balanced"
    ))
])

pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('pca', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",4
,"copy copy: bool, default=TrueIf False, data passed to fit are overwritten and runningfit(X).transform(X) will not yield the expected results,use fit_transform(X) instead.",True
,"whiten whiten: bool, default=FalseWhen True (False by default) the `components_` vectors are multipliedby the square root of n_samples and then divided by the singular valuesto ensure uncorrelated outputs with unit component-wise variances.Whitening will remove some information from the transformed signal(the relative variance scales of the components) but can sometimeimprove the predictive accuracy of the downstream estimators bymaking their data respect some hard-wired assumptions.",False
,"svd_solver svd_solver: {'auto', 'fu

In [9]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=SHUFFLE,
    random_state=RANDOM_STATE
)

cv

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

In [10]:
scoring = {
    "accuracy": "accuracy",
    "precision_weighted": "precision_weighted",
    "recall_weighted": "recall_weighted",
    "f1_weighted": "f1_weighted",
    
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
    
    "mcc": make_scorer(matthews_corrcoef)
}

In [11]:
cv_results = cross_validate(
    estimator=pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

cv_results.keys()

dict_keys(['fit_time', 'score_time', 'test_accuracy', 'test_precision_weighted', 'test_recall_weighted', 'test_f1_weighted', 'test_precision_macro', 'test_recall_macro', 'test_f1_macro', 'test_mcc'])

In [12]:
df_folds = pd.DataFrame({
    "fold": np.arange(1, N_SPLITS + 1),

    "accuracy": cv_results["test_accuracy"],

    "precision_weighted": cv_results["test_precision_weighted"],
    "recall_weighted": cv_results["test_recall_weighted"],
    "f1_weighted": cv_results["test_f1_weighted"],

    "precision_macro": cv_results["test_precision_macro"],
    "recall_macro": cv_results["test_recall_macro"],
    "f1_macro": cv_results["test_f1_macro"],

    "mcc": cv_results["test_mcc"],

    "fit_time": cv_results["fit_time"],
    "score_time": cv_results["score_time"]
})

df_folds

,fold,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro,mcc,fit_time,score_time
0,1,0.831765,0.853223,0.831765,0.830474,0.841580,0.823403,0.820146,0.820886,30.359616,26.315570
1,2,0.831500,0.852720,0.831500,0.830407,0.840562,0.822892,0.819728,0.820562,31.407679,26.919678
2,3,0.831007,0.850129,0.831007,0.829849,0.837702,0.821864,0.818780,0.819812,31.358386,27.818356
3,4,0.828958,0.850434,0.828958,0.827911,0.838538,0.820548,0.817378,0.817844,31.809855,26.505884
4,5,0.826037,0.845766,0.826037,0.824445,0.834013,0.817710,0.813979,0.814653,31.500989,25.405305


In [13]:
summary_cv = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset": str(RUTA_DATASET / NOMBRE_DATASET_TRAIN),
    "shape_dataset": {
        "rows": int(df_train.shape[0]),
        "cols": int(df_train.shape[1])
    },
    "parametros": {
        "label_col": LABEL_COL,
        "n_splits": N_SPLITS,
        "shuffle": SHUFFLE,
        "random_state": RANDOM_STATE,
        "n_components_pca": N_COMPONENTS_PCA,
        "modelo": "SVM",
        "C": C,
        "kernel": KERNEL,
        "gamma": GAMMA,
        "dataset_balanceado": CARPETA_DATASET
    },
    "metricas_media": {
        "accuracy": float(df_folds["accuracy"].mean()),

        "precision_weighted": float(df_folds["precision_weighted"].mean()),
        "recall_weighted": float(df_folds["recall_weighted"].mean()),
        "f1_weighted": float(df_folds["f1_weighted"].mean()),

        "precision_macro": float(df_folds["precision_macro"].mean()),
        "recall_macro": float(df_folds["recall_macro"].mean()),
        "f1_macro": float(df_folds["f1_macro"].mean()),

        "mcc": float(df_folds["mcc"].mean()),
        "fit_time": float(df_folds["fit_time"].mean()),
        "score_time": float(df_folds["score_time"].mean())
    },
    "metricas_std": {
        "accuracy": float(df_folds["accuracy"].std(ddof=1)),

        "precision_weighted": float(df_folds["precision_weighted"].std(ddof=1)),
        "recall_weighted": float(df_folds["recall_weighted"].std(ddof=1)),
        "f1_weighted": float(df_folds["f1_weighted"].std(ddof=1)),

        "precision_macro": float(df_folds["precision_macro"].std(ddof=1)),
        "recall_macro": float(df_folds["recall_macro"].std(ddof=1)),
        "f1_macro": float(df_folds["f1_macro"].std(ddof=1)),

        "mcc": float(df_folds["mcc"].std(ddof=1)),
        "fit_time": float(df_folds["fit_time"].std(ddof=1)),
        "score_time": float(df_folds["score_time"].std(ddof=1))
    }
}

summary_cv

{'experimento': 'BCCC17__RUS_SMOTE_ENN__v1__pca4_svm__v1',
 'dataset': '/home/javier/TFG_MODELOS_SIMPLES/02_datasets/processed/BCCC17__RUS_SMOTE_ENN__v1/BCCC17__RUS_SMOTE_ENN__v1__train.csv',
 'shape_dataset': {'rows': 131810, 'cols': 64},
 'parametros': {'label_col': 'LABEL',
  'n_splits': 5,
  'shuffle': True,
  'random_state': 42,
  'n_components_pca': 4,
  'modelo': 'SVM',
  'C': 1.0,
  'kernel': 'rbf',
  'gamma': 'scale',
  'dataset_balanceado': 'BCCC17__RUS_SMOTE_ENN__v1'},
 'metricas_media': {'accuracy': 0.8298535771185798,
  'precision_weighted': 0.8504543420991896,
  'recall_weighted': 0.8298535771185798,
  'f1_weighted': 0.8286172564053542,
  'precision_macro': 0.8384789941007325,
  'recall_macro': 0.8212834789117369,
  'f1_macro': 0.8180022080450572,
  'mcc': 0.8187514049743708,
  'fit_time': 31.287304973602296,
  'score_time': 26.592958641052245},
 'metricas_std': {'accuracy': 0.0024010067324513077,
  'precision_weighted': 0.002953356873167091,
  'recall_weighted': 0.002401

In [14]:
print("========== RESULTADOS CV ==========")
print(f"Accuracy            : {summary_cv['metricas_media']['accuracy']:.6f} ± {summary_cv['metricas_std']['accuracy']:.6f}")
print()

print(f"Precision weighted  : {summary_cv['metricas_media']['precision_weighted']:.6f} ± {summary_cv['metricas_std']['precision_weighted']:.6f}")
print(f"Recall weighted     : {summary_cv['metricas_media']['recall_weighted']:.6f} ± {summary_cv['metricas_std']['recall_weighted']:.6f}")
print(f"F1 weighted         : {summary_cv['metricas_media']['f1_weighted']:.6f} ± {summary_cv['metricas_std']['f1_weighted']:.6f}")
print()

print(f"Precision macro     : {summary_cv['metricas_media']['precision_macro']:.6f} ± {summary_cv['metricas_std']['precision_macro']:.6f}")
print(f"Recall macro        : {summary_cv['metricas_media']['recall_macro']:.6f} ± {summary_cv['metricas_std']['recall_macro']:.6f}")
print(f"F1 macro            : {summary_cv['metricas_media']['f1_macro']:.6f} ± {summary_cv['metricas_std']['f1_macro']:.6f}")
print()

print(f"MCC                 : {summary_cv['metricas_media']['mcc']:.6f} ± {summary_cv['metricas_std']['mcc']:.6f}")
print()
print(f"Fit time medio      : {summary_cv['metricas_media']['fit_time']:.6f}")
print(f"Score time medio    : {summary_cv['metricas_media']['score_time']:.6f}")

========== RESULTADOS CV ==========
Accuracy            : 0.829854 ± 0.002401

Precision weighted  : 0.850454 ± 0.002953
Recall weighted     : 0.829854 ± 0.002401
F1 weighted         : 0.828617 ± 0.002553

Precision macro     : 0.838479 ± 0.002937
Recall macro        : 0.821283 ± 0.002277
F1 macro            : 0.818002 ± 0.002488

MCC                 : 0.818751 ± 0.002578

Fit time medio      : 31.287305
Score time medio    : 26.592959


In [15]:
ruta_cv_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_CSV
df_folds.to_csv(ruta_cv_csv, index=False)

print("Resultados por fold guardados en:")
print(ruta_cv_csv.resolve())

Resultados por fold guardados en:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/BCCC17__RUS_SMOTE_ENN__v1__pca4_svm__v1/BCCC17__RUS_SMOTE_ENN__v1__pca4_svm__v1__folds.csv


In [16]:
ruta_cv_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_JSON

with open(ruta_cv_json, "w", encoding="utf-8") as f:
    json.dump(summary_cv, f, indent=4, ensure_ascii=False)

print("Resumen CV guardado en:")
print(ruta_cv_json.resolve())

Resumen CV guardado en:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/BCCC17__RUS_SMOTE_ENN__v1__pca4_svm__v1/BCCC17__RUS_SMOTE_ENN__v1__pca4_svm__v1__summary_cv.json


In [17]:
df_folds

,fold,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro,mcc,fit_time,score_time
0,1,0.831765,0.853223,0.831765,0.830474,0.841580,0.823403,0.820146,0.820886,30.359616,26.315570
1,2,0.831500,0.852720,0.831500,0.830407,0.840562,0.822892,0.819728,0.820562,31.407679,26.919678
2,3,0.831007,0.850129,0.831007,0.829849,0.837702,0.821864,0.818780,0.819812,31.358386,27.818356
3,4,0.828958,0.850434,0.828958,0.827911,0.838538,0.820548,0.817378,0.817844,31.809855,26.505884
4,5,0.826037,0.845766,0.826037,0.824445,0.834013,0.817710,0.813979,0.814653,31.500989,25.405305


In [18]:
df_test = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TEST,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset test:")
print(df_test.shape)

df_test.head()

Forma del dataset test:
(472740, 64)


,DST_PORT,PROTOCOL,DURATION,PACKETS_COUNT,FWD_TOTAL_PAYLOAD_BYTES,PAYLOAD_BYTES_MAX,PAYLOAD_BYTES_MIN,PAYLOAD_BYTES_MEAN,PAYLOAD_BYTES_VARIANCE,FWD_PAYLOAD_BYTES_VARIANCE,...,BWD_SYN_FLAG_COUNTS,BWD_CWR_FLAG_COUNTS,BWD_RST_FLAG_COUNTS,PACKETS_IAT_MEAN,FWD_PACKETS_IAT_MEAN,FWD_PACKETS_IAT_STD,BWD_PACKETS_IAT_MEAN,SUBFLOW_FWD_PACKETS,SUBFLOW_FWD_BYTES,LABEL
0,443,0,0.000000,1,0,0,0,0.0,0.00,0.0,...,0,0,0,1.499195e+09,1.499195e+09,0.0,0.000000e+00,0.0,0.0,0
1,53,1,0.741115,4,62,61,31,46.0,225.00,0.0,...,0,0,0,2.470384e-01,2.860000e-06,0.0,4.721000e-05,0.0,0.0,0
2,53,1,0.024214,4,82,124,41,82.5,1722.25,0.0,...,0,0,0,8.071340e-03,3.100000e-06,0.0,3.100000e-06,0.0,0.0,0
3,3737,0,0.000048,2,0,0,0,0.0,0.00,0.0,...,0,0,1,4.816000e-05,1.499451e+09,0.0,1.499451e+09,0.0,0.0,2
4,389,0,0.000048,2,0,0,0,0.0,0.00,0.0,...,0,0,0,4.792000e-05,1.499436e+09,0.0,1.499436e+09,0.0,0.0,0


In [19]:
if LABEL_COL not in df_test.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en test")

print("Última columna test:", df_test.columns[-1])
print("Tipo de LABEL test:", df_test[LABEL_COL].dtype)
print()

print("Distribución de clases en test:")
display(df_test[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna test: LABEL
Tipo de LABEL test: int64

Distribución de clases en test:


,count
LABEL,
0,343010
1,69229
2,32265
3,19146
4,1906
5,1673
6,1371
7,1190
8,1102


In [20]:
X_test = df_test.drop(columns=[LABEL_COL]).copy()
y_test = df_test[LABEL_COL].copy()

print("Shape X_test:", X_test.shape)
print("Shape y_test:", y_test.shape)

Shape X_test: (472740, 63)
Shape y_test: (472740,)


In [21]:
columnas_no_numericas_test = X_test.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_test:")
print(columnas_no_numericas_test)

if len(columnas_no_numericas_test) > 0:
    raise ValueError("Hay columnas no numéricas en X_test. Revísalas antes de seguir.")

Columnas no numéricas en X_test:
[]


In [22]:
pipeline.fit(X_train, y_train)

print("Modelo final entrenado con todo el dataset train.")

Modelo final entrenado con todo el dataset train.


In [23]:
y_pred_test = pipeline.predict(X_test)

print("Predicciones en test generadas.")
print("Número de predicciones:", len(y_pred_test))

Predicciones en test generadas.
Número de predicciones: 472740


In [24]:
metricas_test = {
    "accuracy": accuracy_score(y_test, y_pred_test),

    "precision_weighted": precision_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "recall_weighted": recall_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "f1_weighted": f1_score(y_test, y_pred_test, average="weighted", zero_division=0),

    "precision_macro": precision_score(y_test, y_pred_test, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred_test, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred_test, average="macro", zero_division=0),

    "mcc": matthews_corrcoef(y_test, y_pred_test)
}

metricas_test

{'accuracy': 0.7139442399627702,
 'precision_weighted': 0.9285562993523513,
 'recall_weighted': 0.7139442399627702,
 'f1_weighted': 0.7863294759362074,
 'precision_macro': 0.33205153244682334,
 'recall_macro': 0.7484495872452958,
 'f1_macro': 0.3658256198111198,
 'mcc': 0.5991464498038362}

In [25]:
print("========== RESULTADOS TEST ==========")
print(f"Accuracy            : {metricas_test['accuracy']:.6f}")
print()

print(f"Precision weighted  : {metricas_test['precision_weighted']:.6f}")
print(f"Recall weighted     : {metricas_test['recall_weighted']:.6f}")
print(f"F1 weighted         : {metricas_test['f1_weighted']:.6f}")
print()

print(f"Precision macro     : {metricas_test['precision_macro']:.6f}")
print(f"Recall macro        : {metricas_test['recall_macro']:.6f}")
print(f"F1 macro            : {metricas_test['f1_macro']:.6f}")
print()

print(f"MCC                 : {metricas_test['mcc']:.6f}")

========== RESULTADOS TEST ==========
Accuracy            : 0.713944

Precision weighted  : 0.928556
Recall weighted     : 0.713944
F1 weighted         : 0.786329

Precision macro     : 0.332052
Recall macro        : 0.748450
F1 macro            : 0.365826

MCC                 : 0.599146


In [26]:
labels_ordenadas = sorted(pd.unique(pd.concat([y_test, pd.Series(y_pred_test)])))

cm = confusion_matrix(y_test, y_pred_test, labels=labels_ordenadas)
df_cm = pd.DataFrame(cm, index=labels_ordenadas, columns=labels_ordenadas)

print("Matriz de confusión en test:")
display(df_cm)

Matriz de confusión en test:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,219153,24091,2013,46,7122,15853,6922,7445,28020,530,3845,13990,9670,4310
1,193,60344,88,3724,85,3999,13,0,754,26,0,0,0,3
2,179,16,31586,0,221,0,0,0,17,17,20,22,90,97
3,34,116,0,18975,0,21,0,0,0,0,0,0,0,0
4,6,0,7,0,1882,8,1,1,1,0,0,0,0,0
5,32,196,0,45,103,1216,44,4,23,1,0,0,4,5
6,39,141,0,0,211,4,963,2,0,7,0,0,1,3
7,10,0,2,0,6,0,3,1163,1,0,0,3,2,0
8,4,0,0,0,0,7,0,0,1091,0,0,0,0,0
9,20,103,0,0,99,2,37,4,0,727,31,0,1,0


In [27]:
print("========== CLASSIFICATION REPORT TEST ==========")
print(classification_report(y_test, y_pred_test, zero_division=0))

========== CLASSIFICATION REPORT TEST ==========
              precision    recall  f1-score   support

           0       1.00      0.64      0.78    343010
           1       0.71      0.87      0.78     69229
           2       0.94      0.98      0.96     32265
           3       0.83      0.99      0.90     19146
           4       0.19      0.99      0.32      1906
           5       0.06      0.73      0.11      1673
           6       0.12      0.70      0.21      1371
           7       0.13      0.98      0.24      1190
           8       0.04      0.99      0.07      1102
           9       0.56      0.71      0.62      1024
          10       0.06      0.49      0.11       546
          11       0.01      0.51      0.02       271
          12       0.00      0.40      0.00         5
          13       0.00      0.50      0.00         2

    accuracy                           0.71    472740
   macro avg       0.33      0.75      0.37    472740
weighted avg       0.93      0.

In [28]:
summary_test = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset_test": str(RUTA_DATASET / NOMBRE_DATASET_TEST),
    "shape_test": {
        "rows": int(df_test.shape[0]),
        "cols": int(df_test.shape[1])
    },
    "parametros": {
        "label_col": LABEL_COL,
        "n_splits": N_SPLITS,
        "shuffle": SHUFFLE,
        "random_state": RANDOM_STATE,
        "n_components_pca": N_COMPONENTS_PCA,
        "modelo": "SVM",
        "C": C,
        "kernel": KERNEL,
        "gamma": GAMMA,
        "dataset_balanceado": CARPETA_DATASET
    },
    "metricas_test": {
        "accuracy": float(metricas_test["accuracy"]),

        "precision_weighted": float(metricas_test["precision_weighted"]),
        "recall_weighted": float(metricas_test["recall_weighted"]),
        "f1_weighted": float(metricas_test["f1_weighted"]),

        "precision_macro": float(metricas_test["precision_macro"]),
        "recall_macro": float(metricas_test["recall_macro"]),
        "f1_macro": float(metricas_test["f1_macro"]),

        "mcc": float(metricas_test["mcc"])
    }
}

summary_test

{'experimento': 'BCCC17__RUS_SMOTE_ENN__v1__pca4_svm__v1',
 'dataset_test': '/home/javier/TFG_MODELOS_SIMPLES/02_datasets/processed/BCCC17__RUS_SMOTE_ENN__v1/BCCC17__RUS_SMOTE_ENN__v1__test.csv',
 'shape_test': {'rows': 472740, 'cols': 64},
 'parametros': {'label_col': 'LABEL',
  'n_splits': 5,
  'shuffle': True,
  'random_state': 42,
  'n_components_pca': 4,
  'modelo': 'SVM',
  'C': 1.0,
  'kernel': 'rbf',
  'gamma': 'scale',
  'dataset_balanceado': 'BCCC17__RUS_SMOTE_ENN__v1'},
 'metricas_test': {'accuracy': 0.7139442399627702,
  'precision_weighted': 0.9285562993523513,
  'recall_weighted': 0.7139442399627702,
  'f1_weighted': 0.7863294759362074,
  'precision_macro': 0.33205153244682334,
  'recall_macro': 0.7484495872452958,
  'f1_macro': 0.3658256198111198,
  'mcc': 0.5991464498038362}}

In [29]:
df_metricas_test = pd.DataFrame([metricas_test])

ruta_test_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CSV
df_metricas_test.to_csv(ruta_test_csv, index=False)

print("Métricas test guardadas en:")
print(ruta_test_csv.resolve())

Métricas test guardadas en:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/BCCC17__RUS_SMOTE_ENN__v1__pca4_svm__v1/BCCC17__RUS_SMOTE_ENN__v1__pca4_svm__v1__metricas_test.csv


In [30]:
ruta_cm_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CM_CSV
df_cm.to_csv(ruta_cm_csv, index=True)

print("Matriz de confusión test guardada en:")
print(ruta_cm_csv.resolve())

Matriz de confusión test guardada en:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/BCCC17__RUS_SMOTE_ENN__v1__pca4_svm__v1/BCCC17__RUS_SMOTE_ENN__v1__pca4_svm__v1__confusion_matrix_test.csv


In [31]:
ruta_test_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_JSON

with open(ruta_test_json, "w", encoding="utf-8") as f:
    json.dump(summary_test, f, indent=4, ensure_ascii=False)

print("Resumen test guardado en:")
print(ruta_test_json.resolve())

Resumen test guardado en:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/BCCC17__RUS_SMOTE_ENN__v1__pca4_svm__v1/BCCC17__RUS_SMOTE_ENN__v1__pca4_svm__v1__summary_test.json


In [32]:
print("========== RESUMEN FINAL ==========")
print("CV:")
print(summary_cv["metricas_media"])
print()
print("TEST:")
print(summary_test["metricas_test"])

========== RESUMEN FINAL ==========
CV:
{'accuracy': 0.8298535771185798, 'precision_weighted': 0.8504543420991896, 'recall_weighted': 0.8298535771185798, 'f1_weighted': 0.8286172564053542, 'precision_macro': 0.8384789941007325, 'recall_macro': 0.8212834789117369, 'f1_macro': 0.8180022080450572, 'mcc': 0.8187514049743708, 'fit_time': 31.287304973602296, 'score_time': 26.592958641052245}

TEST:
{'accuracy': 0.7139442399627702, 'precision_weighted': 0.9285562993523513, 'recall_weighted': 0.7139442399627702, 'f1_weighted': 0.7863294759362074, 'precision_macro': 0.33205153244682334, 'recall_macro': 0.7484495872452958, 'f1_macro': 0.3658256198111198, 'mcc': 0.5991464498038362}
